# 긴 문맥 재정렬(Long-context reordering)

검색 순위가 높은 문서를 프롬프트의 앞과 뒤에 번갈아 배치해, 중요한 정보가
긴 문맥 한가운데 묻히는 현상을 완화합니다. 보관된 transformer 클래스 대신
짧고 검증 가능한 순수 함수를 Runnable로 조합합니다.

논문: [Lost in the Middle](https://arxiv.org/abs/2307.03172)


> **2026-09-21 업데이트**
>
> 이 노트북은 `langchain 1.4.2`, `langchain-core 1.6.3`,
> `langchain-openai 1.6.2`, `langchain-chroma 1.1.0` 기준으로 다시 작성했습니다.
> LangChain v1에서 예전 `langchain.retrievers` 구현은 `langchain-classic`으로
> 이동했고 `langchain-community`도 보관 상태이므로, 새 코드에서는 두 패키지와
> `langchain-teddynote`에 의존하지 않습니다. 대신 `langchain-core`의 Runnable,
> 공급자별 파트너 패키지, 명시적인 검색 함수를 조합합니다.
>
> 공식 참고: [LangChain v1 변경 사항](https://docs.langchain.com/oss/python/releases/langchain-v1),
> [v1 마이그레이션](https://docs.langchain.com/oss/python/migrate/langchain-v1),
> [OpenAI 임베딩](https://docs.langchain.com/oss/python/integrations/embeddings/openai),
> [Chroma 통합](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma)


In [ ]:
# 최초 1회만 주석을 해제하세요.
# %pip install -qU "langchain==1.4.2" "langchain-core==1.6.3" \
#   "langchain-openai==1.6.2" "langchain-chroma==1.1.0" python-dotenv


In [ ]:
import getpass
import os
from operator import itemgetter
from uuid import uuid4

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_openai import OpenAIEmbeddings

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

CHROMA_CONFIGURATION = {"hnsw": {"space": "cosine"}}


In [ ]:
texts = [
    "이건 그냥 내가 아무렇게나 적어본 글입니다.",
    "사용자와 대화하도록 설계된 AI인 ChatGPT는 다양한 질문에 답할 수 있습니다.",
    "아이폰, 아이패드, 맥북은 애플의 대표 제품입니다.",
    "ChatGPT는 OpenAI가 개발했으며 지속적으로 개선되고 있습니다.",
    "ChatGPT는 질문을 이해하고 답변을 생성하도록 학습되었습니다.",
    "애플 워치와 에어팟은 애플의 웨어러블 제품군입니다.",
    "ChatGPT는 문제 해결과 창의적인 아이디어 생성에도 활용됩니다.",
    "비트코인은 가치 저장 수단으로도 사용되는 디지털 자산입니다.",
    "ChatGPT의 기능은 모델과 제품 업데이트에 따라 발전합니다.",
    "FIFA 월드컵은 4년마다 열리는 국제 축구 대회입니다.",
]
docs = [
    Document(page_content=text, metadata={"doc_id": f"doc-{index}"})
    for index, text in enumerate(texts)
]
vectorstore = Chroma.from_documents(
    docs,
    OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_name=f"long-context-{uuid4().hex}",
    ids=[doc.metadata["doc_id"] for doc in docs],
    collection_configuration=CHROMA_CONFIGURATION,
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
query = "ChatGPT에 대해 무엇을 말해줄 수 있나요?"
ranked_docs = retriever.invoke(query)


## 재정렬 함수

입력은 관련도 내림차순이어야 합니다. 1위는 맨 앞, 2위는 맨 뒤, 3위는
앞쪽 두 번째처럼 배치하고 낮은 순위일수록 가운데로 보냅니다.


In [ ]:
def long_context_reorder(documents: list[Document]) -> list[Document]:
    # 1위, 3위, 5위 ... 는 앞쪽에 두고
    # 2위, 4위, 6위 ... 는 뒤에서부터 배치합니다.
    # 결과적으로 1위는 맨 앞, 2위는 맨 뒤에 놓입니다.
    return documents[::2] + list(reversed(documents[1::2]))


reordered_docs = long_context_reorder(ranked_docs)
print("원래 순위:", [doc.metadata["doc_id"] for doc in ranked_docs])
print("재정렬 후:", [doc.metadata["doc_id"] for doc in reordered_docs])


In [ ]:
def format_docs(documents: list[Document]) -> str:
    return "\n\n".join(
        f"[{index}] {doc.page_content}\n[source: {doc.metadata['doc_id']}]"
        for index, doc in enumerate(documents, start=1)
    )


reorder_and_format = (
    retriever
    | RunnableLambda(long_context_reorder)
    | RunnableLambda(format_docs)
)
print(reorder_and_format.invoke(query))


## 재정렬을 포함한 RAG 체인


In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "제공된 문맥만 사용해 답하세요. 근거가 없으면 모른다고 답하세요. "
            "답변 언어는 {language}입니다.\n\n<context>\n{context}\n</context>",
        ),
        ("user", "{question}"),
    ]
)
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.4-mini")
model = init_chat_model(f"openai:{model_name}", temperature=0)

chain = (
    {
        "context": itemgetter("question") | reorder_and_format,
        "question": itemgetter("question"),
        "language": itemgetter("language"),
    }
    | prompt
    | model
    | StrOutputParser()
)
answer = chain.invoke({"question": query, "language": "한국어"})
print(answer)
